# VRTPP-PR Scalability Experiment

10 random problem instances per paper Section VI methodology.
Orbital elements: a in [1,3] AU, e in [0,0.3], i in [0,5] deg, Omega/omega/M in [0,360] deg.

**To run a different experiment:** change `N_R` and `N_M` in the Config cell below, then Run All.

## Imports

In [1]:
import numpy as np
from scipy.optimize import minimize, Bounds
from typing import Dict, List, Tuple, Optional
from dataclasses import dataclass
import time
import warnings
warnings.filterwarnings('ignore')

# Optimization
import gurobipy as gp
from gurobipy import GRB

## Orbital Mechanics

In [2]:
class OrbitalBody:
    """Celestial body with orbital elements."""

    def __init__(self, name: str, a: float, e: float, i: float,
                 Omega: float, omega: float, M0: float, epoch: float = 0.0):
        """
        Parameters:
        -----------
        name : str - Body name
        a : float - Semi-major axis [AU]
        e : float - Eccentricity
        i : float - Inclination [degrees]
        Omega : float - RAAN [degrees]
        omega : float - Argument of periapsis [degrees]
        M0 : float - Mean anomaly at epoch [degrees]
        epoch : float - Reference epoch [TU]
        """
        self.name = name
        self.a = a
        self.e = e
        self.i = np.deg2rad(i)
        self.Omega = np.deg2rad(Omega)
        self.omega = np.deg2rad(omega)
        self.M0 = np.deg2rad(M0)
        self.epoch = epoch

    def position_at_time(self, t: float, mu: float = 1.0) -> np.ndarray:
        """Calculate position vector at time t using Kepler's equation."""
        n = np.sqrt(mu / self.a**3)
        M = self.M0 + n * (t - self.epoch)
        E = self._solve_kepler(M, self.e)
        nu = 2 * np.arctan2(np.sqrt(1 + self.e) * np.sin(E / 2),
                            np.sqrt(1 - self.e) * np.cos(E / 2))
        r_mag = self.a * (1 - self.e * np.cos(E))
        x_orb = r_mag * np.cos(nu)
        y_orb = r_mag * np.sin(nu)
        R = self._rotation_matrix()
        r_orb = np.array([x_orb, y_orb, 0])
        r = R @ r_orb
        return r

    def velocity_at_time(self, t: float, mu: float = 1.0) -> np.ndarray:
        """Calculate velocity vector at time t."""
        n = np.sqrt(mu / self.a**3)
        M = self.M0 + n * (t - self.epoch)
        E = self._solve_kepler(M, self.e)
        nu = 2 * np.arctan2(np.sqrt(1 + self.e) * np.sin(E / 2),
                            np.sqrt(1 - self.e) * np.cos(E / 2))
        h = np.sqrt(mu * self.a * (1 - self.e**2))
        vx_orb = -(mu / h) * np.sin(nu)
        vy_orb = (mu / h) * (self.e + np.cos(nu))
        R = self._rotation_matrix()
        v_orb = np.array([vx_orb, vy_orb, 0])
        v = R @ v_orb
        return v

    def _solve_kepler(self, M: float, e: float, tol: float = 1e-10) -> float:
        """Solve Kepler's equation using Newton-Raphson."""
        E = M if e < 0.8 else np.pi
        for _ in range(50):
            f = E - e * np.sin(E) - M
            f_prime = 1 - e * np.cos(E)
            E_new = E - f / f_prime
            if abs(E_new - E) < tol:
                return E_new
            E = E_new
        return E

    def _rotation_matrix(self) -> np.ndarray:
        """Compute rotation matrix from orbital plane to heliocentric frame."""
        c_O, s_O = np.cos(self.Omega), np.sin(self.Omega)
        c_i, s_i = np.cos(self.i), np.sin(self.i)
        c_w, s_w = np.cos(self.omega), np.sin(self.omega)
        R = np.array([
            [c_O * c_w - s_O * c_i * s_w, -c_O * s_w - s_O * c_i * c_w, s_O * s_i],
            [s_O * c_w + c_O * c_i * s_w, -s_O * s_w + c_O * c_i * c_w, -c_O * s_i],
            [s_i * s_w, s_i * c_w, c_i]
        ])
        return R

In [3]:
class LambertSolver:
    """Robust Lambert solver using universal variables with Stumpff functions."""

    def __init__(self, mu: float = 1.0):
        self.mu = mu

    def solve(self, r1_vec: np.ndarray, r2_vec: np.ndarray, tof: float,
              prograde: bool = True) -> Tuple[np.ndarray, np.ndarray]:
        """Solve Lambert's problem."""
        r1 = np.linalg.norm(r1_vec)
        r2 = np.linalg.norm(r2_vec)

        cos_dnu = np.dot(r1_vec, r2_vec) / (r1 * r2)
        cos_dnu = np.clip(cos_dnu, -1.0, 1.0)

        cross = np.cross(r1_vec, r2_vec)
        if prograde:
            if cross[2] >= 0:
                dnu = np.arccos(cos_dnu)
            else:
                dnu = 2 * np.pi - np.arccos(cos_dnu)
        else:
            if cross[2] < 0:
                dnu = np.arccos(cos_dnu)
            else:
                dnu = 2 * np.pi - np.arccos(cos_dnu)

        A = np.sin(dnu) * np.sqrt(r1 * r2 / (1 - cos_dnu))

        if abs(A) < 1e-14:
            raise ValueError("Degenerate Lambert problem")

        # Stumpff functions
        def C2(psi):
            if psi > 1e-6:
                return (1 - np.cos(np.sqrt(psi))) / psi
            elif psi < -1e-6:
                return (np.cosh(np.sqrt(-psi)) - 1) / (-psi)
            else:
                return 1.0 / 2.0

        def C3(psi):
            if psi > 1e-6:
                sp = np.sqrt(psi)
                return (sp - np.sin(sp)) / (psi * sp)
            elif psi < -1e-6:
                sp = np.sqrt(-psi)
                return (np.sinh(sp) - sp) / ((-psi) * sp)
            else:
                return 1.0 / 6.0

        # Newton-Raphson iteration with bisection fallback
        psi_n = 0.0
        psi_up = 4 * np.pi**2
        psi_low = -4 * np.pi**2

        for _ in range(100):
            c2 = C2(psi_n)
            c3 = C3(psi_n)

            y_n = r1 + r2 + A * (psi_n * c3 - 1) / np.sqrt(c2)

            if y_n < 0:
                # Readjust psi until y_n is non-negative (with iteration limit)
                for _ in range(2000):
                    psi_n += 0.1
                    c2 = C2(psi_n)
                    c3 = C3(psi_n)
                    y_n = r1 + r2 + A * (psi_n * c3 - 1) / np.sqrt(c2)
                    if y_n >= 0:
                        break
                else:
                    raise ValueError("Lambert solver: could not find valid y_n (geometry may be near-degenerate)")

            chi = np.sqrt(y_n / c2)

            tof_n = (chi**3 * c3 + A * np.sqrt(y_n)) / np.sqrt(self.mu)

            if abs(tof_n - tof) < 1e-8 * abs(tof):
                break

            if tof_n <= tof:
                psi_low = psi_n
            else:
                psi_up = psi_n

            # Newton step with bisection guard
            dtof_dpsi = (chi**3 * (C3(psi_n) - 3 * c3 * C2(psi_n) / (2 * c2)) / (2 * c2) +
                         (A / 8) * (3 * c3 * np.sqrt(y_n) / c2 + A / chi))
            dtof_dpsi /= np.sqrt(self.mu)

            if abs(dtof_dpsi) > 1e-14:
                psi_new = psi_n + (tof - tof_n) / dtof_dpsi
                if psi_low <= psi_new <= psi_up:
                    psi_n = psi_new
                else:
                    psi_n = (psi_up + psi_low) / 2
            else:
                psi_n = (psi_up + psi_low) / 2

        f = 1 - y_n / r1
        g_dot = 1 - y_n / r2
        g = A * np.sqrt(y_n / self.mu)

        if abs(g) < 1e-14:
            raise ValueError("Lambert solver: g is near zero")

        v1 = (r2_vec - f * r1_vec) / g
        v2 = (g_dot * r2_vec - r1_vec) / g

        return v1, v2


## Earth Orbital Elements

In [4]:
earth = OrbitalBody(
    name="Earth",
    a=1.0009, e=0.0173, i=0.0032,
    Omega=171.7283, omega=289.5838, M0=318.5855,
    epoch=0.0
)

## Parameters

In [5]:
@dataclass
class Parameters:
    """Problem parameters from Table 2."""

    # Physical constants
    mu_sun: float = 1.0       # Gravitational parameter [AU^3/TU^2] (canonical)
    mu_earth: float = 3.986e5  # km^3/s^2
    g0: float = 9.81e-3       # km/s^2

    # Spacecraft
    m_dry: float = 300.0      # kg
    m_max: float = 20000.0    # kg
    q_max: float = 30.0       # kg
    I_sp: float = 457.0       # s

    # Problem size
    n_bv: int = 3             # Max spacecraft
    n_rv: int = 3             # Max refueling visits

    # Mission
    T_service: float = 2.0 / 58.132  # days to TU
    lambda_weight: float = 5e-5

    # Profit and mining (from case study)
    profit: float = 10.0      # Same for all
    mining_mass: float = 10.0  # kg, same for all

    # Parking orbit
    r0_park: float = 7000.0   # km

    # Unit conversions
    AU_to_km: float = 1.496e8
    TU_to_sec: float = 58.132 * 86400


params = Parameters()

print(f"Spacecraft:")
print(f"  Dry mass:    {params.m_dry} kg")
print(f"  Max mass:    {params.m_max} kg")
print(f"  Isp:         {params.I_sp} s")
print(f"\nMission:")
print(f"  Profit/asteroid: {params.profit}")
print(f"  Mining/asteroid: {params.mining_mass} kg")
print(f"  Lambda:          {params.lambda_weight}")

Spacecraft:
  Dry mass:    300.0 kg
  Max mass:    20000.0 kg
  Isp:         457.0 s

Mission:
  Profit/asteroid: 10.0
  Mining/asteroid: 10.0 kg
  Lambda:          5e-05


## Index Sets & Node Mapping

In [6]:
def build_index_sets(params: Parameters, n_refuel: int, n_mine: int) -> Dict:
    """Build index sets (Equations 1-9)."""

    n_bv = params.n_bv
    n_rv = params.n_rv

    B0 = [0]
    Bv = list(range(1, n_bv + 1))
    Bs = list(range(0, n_bv + 1))
    Be = list(range(n_bv + 1, 2 * n_bv + 2))  # Fixed: start at n_bv+1 to avoid overlap with Bs

    R0 = list(range(2 * n_bv + 2, 2 * n_bv + n_refuel + 2))
    Rv = list(range(2 * n_bv + n_refuel + 2, 2 * n_bv + n_refuel * n_rv + 2))
    R = R0 + Rv

    M = list(range(2 * n_bv + n_refuel * n_rv + 2,
                   2 * n_bv + n_refuel * n_rv + n_mine + 2))

    V = R + M
    N = Bs + Be + V

    k_prime = {k: k + n_bv + 1 for k in Bs}  # Fixed: offset by n_bv+1 to match corrected Be

    return {
        'B0': B0, 'Bv': Bv, 'Bs': Bs, 'Be': Be,
        'R0': R0, 'Rv': Rv, 'R': R, 'M': M, 'V': V, 'N': N,
        'k_prime': k_prime
    }


def build_node_mapping(sets: Dict, refueling_bodies: List, mining_bodies: List) -> Tuple[Dict, Dict]:
    """Map node indices to celestial bodies."""

    node_to_body = {}
    node_to_name = {}

    # Bases (starting and ending -- both Earth)
    for node in sets['Bs'] + sets['Be']:
        node_to_body[node] = earth
        node_to_name[node] = "Earth"

    # Refueling (including virtual)
    for i, node in enumerate(sets['R']):
        original_idx = i % len(refueling_bodies)
        node_to_body[node] = refueling_bodies[original_idx]
        node_to_name[node] = refueling_bodies[original_idx].name

    # Mining
    for i, node in enumerate(sets['M']):
        node_to_body[node] = mining_bodies[i]
        node_to_name[node] = mining_bodies[i].name

    return node_to_body, node_to_name


## Random Instance Generator

In [7]:
import random

def generate_random_asteroids(n_r, n_m, seed=None):
    rng = random.Random(seed)
    def rand_body(name):
        return OrbitalBody(
            name=name,
            a=rng.uniform(1.0, 3.0),
            e=rng.uniform(0.0, 0.3),
            i=rng.uniform(0.0, 5.0),
            Omega=rng.uniform(0.0, 360.0),
            omega=rng.uniform(0.0, 360.0),
            M0=rng.uniform(0.0, 360.0),
        )
    refueling = [rand_body("R" + str(k+1)) for k in range(n_r)]
    mining    = [rand_body("M" + str(k+1)) for k in range(n_m)]
    return refueling, mining

## Trajectory Optimizer (NLP)

In [8]:
class TrajectoryOptimizer:
    """Optimizes trajectory for a single segment using Lambert's problem."""

    def __init__(self, params: Parameters):
        self.params = params
        self.lambert = LambertSolver(mu=params.mu_sun)

    def compute_delta_v(self, body_i: OrbitalBody, body_j: OrbitalBody,
                        T_d: float, T_t: float) -> float:
        """
        Compute delta-v for a transfer.

        Returns delta-v in km/s.
        """
        # Get positions and velocities
        r1 = body_i.position_at_time(T_d, self.params.mu_sun)
        r2 = body_j.position_at_time(T_d + T_t, self.params.mu_sun)
        v1_orbit = body_i.velocity_at_time(T_d, self.params.mu_sun)
        v2_orbit = body_j.velocity_at_time(T_d + T_t, self.params.mu_sun)

        # Solve Lambert's problem
        try:
            v1_transfer, v2_transfer = self.lambert.solve(r1, r2, T_t, prograde=True)
        except Exception:
            return 100.0

        # Convert to km/s
        conversion = self.params.AU_to_km / self.params.TU_to_sec

        # Departure and arrival delta-v in heliocentric frame
        dv1_heli = np.linalg.norm(v1_transfer - v1_orbit) * conversion
        dv2_heli = np.linalg.norm(v2_orbit - v2_transfer) * conversion

        # Add Earth departure/arrival if applicable (Equation 48)
        if body_i.name == "Earth":
            v_inf = (v1_transfer - v1_orbit) * conversion
            dv1 = self._earth_departure_dv(v_inf)
        else:
            dv1 = dv1_heli

        if body_j.name == "Earth":
            v_inf = (v2_orbit - v2_transfer) * conversion
            dv2 = self._earth_arrival_dv(v_inf)
        else:
            dv2 = dv2_heli

        return dv1 + dv2

    def _earth_departure_dv(self, v_inf: np.ndarray) -> float:
        """Compute Earth departure delta-v (Equation 48)."""
        v_inf_mag = np.linalg.norm(v_inf)
        v_park = np.sqrt(self.params.mu_earth / self.params.r0_park)
        v_depart = np.sqrt(v_inf_mag**2 + 2 * self.params.mu_earth / self.params.r0_park)
        return abs(v_depart - v_park)

    def _earth_arrival_dv(self, v_inf: np.ndarray) -> float:
        """Compute Earth arrival delta-v (Equation 48)."""
        v_inf_mag = np.linalg.norm(v_inf)
        v_park = np.sqrt(self.params.mu_earth / self.params.r0_park)
        v_arrive = np.sqrt(v_inf_mag**2 + 2 * self.params.mu_earth / self.params.r0_park)
        return abs(v_arrive - v_park)

    def optimize_segment(self, body_i: OrbitalBody, body_j: OrbitalBody,
                         T_arrival_i: float, T_t_prev: float = None,
                         T_d_prev: float = None) -> Dict:
        """
        Optimize single trajectory segment using trust-region NLP (paper Sec. IV.B.2).

        Fix 1: Warm-starts both T_d and T_t from previous iteration's solution so
               the solver reliably finds the same local minimum.
        Fix 2: Tighter gtol/xtol force the solver to commit to a precise optimum,
               reducing between-call drift that causes convergence oscillation.

        Returns:
        --------
        result : Dict with T_d, T_t, delta_v, T_a, mass_ratio
        """
        from scipy.optimize import Bounds as ScipyBounds
        # Service time (mining/refueling) applies only at asteroid nodes.
        # Earth is the base depot — no service hold before departure (Eq. 44).
        service = self.params.T_service if body_i.name != "Earth" else 0.0
        T_d_min = T_arrival_i + service

        a_transfer = (body_i.a + body_j.a) / 2
        T_t_hoh = np.pi * np.sqrt(a_transfer**3 / self.params.mu_sun)
        if T_t_prev is not None:
            T_t_init = T_t_prev
        else:
            # Fix 13 (revised): scan T_t candidates at T_d_min to land in the right basin.
            # Hohmann T_t is a poor warm-start for eccentric bodies — e.g. FG3->Bennu has
            # two local minima: Hohmann (~3.6 TU) lands at 11.4 km/s; T_t~7 TU finds 7.3 km/s.
            # Scanning 7 candidates at the actual T_d_min (not the init grid's T_d) is
            # contextually correct and costs only 7 Lambert solves per first-seen arc.
            T_t_candidates = np.arange(1.0, 14.0, 2.0)
            best_T_t = T_t_hoh
            best_dv_scan = 1e9
            for T_t_cand in T_t_candidates:
                try:
                    dv_cand = self.compute_delta_v(body_i, body_j, T_d_min, T_t_cand)
                    if np.isfinite(dv_cand) and dv_cand < best_dv_scan:
                        best_dv_scan = dv_cand
                        best_T_t = T_t_cand
                except Exception:
                    pass
            T_t_init = best_T_t

        # Fix 1: warm-start T_d from previous result (clamped to remain feasible)
        # Cap the wait time at each body to 5 TU (~8 months). Without this, the NLP
        # finds low-dv windows 20-35 TU in the future that are physically valid but
        # require years-long stays at asteroids — operationally impossible and a
        # symptom of missing time-window constraints (future feature).
        T_d_max = T_d_min + 5.0
        T_d_init = max(T_d_prev, T_d_min) if T_d_prev is not None else T_d_min
        T_d_init = min(T_d_init, T_d_max)  # clamp warm-start into valid range

        x0 = np.array([T_d_init, max(T_t_init, 1e-5)], dtype=float)
        bounds = ScipyBounds([T_d_min, 1e-5], [T_d_max, 30.0])

        def objective(x):
            T_d, T_t = x
            dv = self.compute_delta_v(body_i, body_j, T_d, T_t)
            return dv if np.isfinite(dv) else 1e6

        try:
            res = minimize(
                objective,
                x0=x0,
                method='trust-constr',
                bounds=bounds,
                # Fix 2: tighter tolerances so solver commits to a precise local min
                options={'maxiter': 500, 'verbose': 0, 'gtol': 1e-8, 'xtol': 1e-8}
            )
            # trust-constr often returns success=False even for valid solutions
            # (gradient tolerance not met). Only check function value.
            success = res is not None and np.isfinite(res.fun) and res.fun < 100.0
        except Exception:
            success = False
            res = None

        if not success:
            return {
                'T_d': T_d_init,
                'T_t': max(T_t_init, 1e-5),
                'delta_v': 100.0,
                'T_a': T_d_init + max(T_t_init, 1e-5),
                'mass_ratio': 1e-10
            }

        T_d_opt, T_t_opt = res.x
        dv_opt = res.fun
        mass_ratio = np.exp(-dv_opt / (self.params.g0 * self.params.I_sp))
        mass_ratio = min(max(mass_ratio, 1e-10), 0.999)

        return {
            'T_d': T_d_opt,
            'T_t': T_t_opt,
            'delta_v': dv_opt,
            'T_a': T_d_opt + T_t_opt,
            'mass_ratio': mass_ratio
        }

## MILP Builder

In [9]:
def build_milp(params: Parameters, sets: Dict, mass_ratios: Dict,
               node_to_name: Dict, node_to_body: Dict) -> Tuple[gp.Model, Dict]:
    """
    Build MILP model with fixed mass ratios.

    Returns model and variables dict.
    """

    model = gp.Model("VRTPP-PR")
    model.setParam('OutputFlag', 0)
    model.setParam('MIPGap', 0.03)  # Accept 3% gap as optimal

    Bs, V, R, M = sets['Bs'], sets['V'], sets['R'], sets['M']
    k_prime = sets['k_prime']

    m_dry = params.m_dry
    m_max = params.m_max
    q_max = params.q_max
    lambda_w = params.lambda_weight
    m_m = params.mining_mass
    p = params.profit

    # Variables
    x, u, q, r, y = {}, {}, {}, {}, {}

    for k in Bs:
        for j in V:
            x[k, k, j] = model.addVar(vtype=GRB.BINARY)
    for k in Bs:
        for i in V:
            for j in V:
                # Filter same physical body: direct Bennu->Bennu (virtual) arcs
                # are physically meaningless and create near-free hops.
                if i != j and node_to_body[i].name != node_to_body[j].name:
                    x[k, i, j] = model.addVar(vtype=GRB.BINARY)
    for k in Bs:
        for i in V:
            x[k, i, k_prime[k]] = model.addVar(vtype=GRB.BINARY)

    for i in Bs + V:
        u[i] = model.addVar(lb=0, ub=m_max)
    for i in V:
        q[i] = model.addVar(lb=0, ub=q_max)
    for i in R:
        r[i] = model.addVar(lb=0)
    for k in Bs:
        for i in V:
            y[k, i] = model.addVar(lb=0, ub=q_max)

    model.update()

    # Objective (Eq. 10)
    profit_term = gp.quicksum(
        p * (gp.quicksum(x[k, i, j] for k in Bs for j in V if i != j) +
             gp.quicksum(x[k, i, k_prime[k]] for k in Bs))
        for i in M
    )
    fuel_term = gp.quicksum(u[k] - m_dry * gp.quicksum(x[k, k, j] for j in V) for k in Bs) + \
                gp.quicksum(r[i] for i in R)

    model.setObjective(profit_term - lambda_w * fuel_term, GRB.MAXIMIZE)

    # Network constraints (Eqs. 11-14)
    for k in Bs:
        model.addConstr(gp.quicksum(x[k, k, j] for j in V) <= 1)
    for j in R:
        model.addConstr(gp.quicksum(x[k, k, j] for k in Bs) +
                        gp.quicksum(x[k, i, j] for k in Bs for i in V if i != j and (k, i, j) in x) <= 1)
    for i in M:
        model.addConstr(gp.quicksum(x[k, i, j] for k in Bs for j in V if i != j) +
                        gp.quicksum(x[k, i, k_prime[k]] for k in Bs) <= 1)
    for j in V:
        for k in Bs:
            model.addConstr(x[k, k, j] - x[k, j, k_prime[k]] +
                            gp.quicksum(x[k, i, j] - x[k, j, i] for i in V if i != j and (k, i, j) in x) == 0)

    # Mass flow (Eqs. 38-42)
    # Upper bounds on arrival mass (mass conservation)
    for k in Bs:
        for j in V:
            if (k, j) in mass_ratios:
                m_kj = mass_ratios[(k, j)]
                model.addConstr(u[j] <= m_kj * u[k] + m_max * (1 - x[k, k, j]))

    for i in M:
        for j in V:
            if i != j and (i, j) in mass_ratios:
                m_ij = mass_ratios[(i, j)]
                if np.isfinite(m_ij) and 0 < m_ij <= 1:
                    model.addConstr(u[j] <= m_ij * (u[i] + m_m) + m_max * (1 - gp.quicksum(x[k, i, j] for k in Bs)))

    for i in R:
        for j in V:
            if i != j and (i, j) in mass_ratios:
                m_ij = mass_ratios[(i, j)]
                if np.isfinite(m_ij) and 0 < m_ij <= 1:
                    model.addConstr(u[j] <= m_ij * (u[i] + r[i]) + m_max * (1 - gp.quicksum(x[k, i, j] for k in Bs)))

    # Eqs. 41-42: Ending-base legs carry mined/refueled cargo (y[k,i] = q[i] on return)
    # Mining nodes -> ending base (Eq. 41)
    for i in M:
        for k in Bs:
            if (i, k_prime[k]) in mass_ratios:
                m_ik = mass_ratios[(i, k_prime[k])]
                model.addConstr(
                    m_dry + y[k, i] <= m_ik * (u[i] + m_m) + m_max * (1 - x[k, i, k_prime[k]])
                )

    # Refueling nodes -> ending base (Eq. 42)
    for i in R:
        for k in Bs:
            if (i, k_prime[k]) in mass_ratios:
                m_ik = mass_ratios[(i, k_prime[k])]
                model.addConstr(
                    m_dry + y[k, i] <= m_ik * (u[i] + r[i]) + m_max * (1 - x[k, i, k_prime[k]])
                )

    # Cumulative mining (Eqs. 20-22)
    for j in M:
        model.addConstr(q[j] >= m_m - q_max * (1 - gp.quicksum(x[k, k, j] for k in Bs)))
    for i in V:
        for j in M:
            if i != j:
                model.addConstr(q[j] >= q[i] + m_m - q_max * (1 - gp.quicksum(x[k, i, j] for k in Bs)))
    for i in V:
        for j in R:
            if i != j:
                model.addConstr(q[j] >= q[i] - q_max * (1 - gp.quicksum(x[k, i, j] for k in Bs if (k, i, j) in x)))

    # Physical limits (Eqs. 23-26)
    for i in M:
        model.addConstr(u[i] >= m_dry + q[i] - m_m)
        model.addConstr(u[i] + m_m <= m_max)
    for i in R:
        model.addConstr(u[i] >= m_dry + q[i])
        model.addConstr(u[i] + r[i] <= m_max)

    # Linearization: y[k,i] = q[i] * x[k,i,k'(k)] (cargo only on ending-base arc)
    # This tightens the paper definition: y_ki = q_i * x^k_{i,k'(k)}
    for k in Bs:
        for i in V:
            model.addConstr(y[k, i] <= q[i])
            model.addConstr(y[k, i] <= q_max * x[k, i, k_prime[k]])
            model.addConstr(y[k, i] >= q[i] - q_max * (1 - x[k, i, k_prime[k]]))

    return model, {'x': x, 'u': u, 'q': q, 'r': r, 'y': y}

## Route Extraction & Mass Ratio Initialization

In [10]:
def extract_routes(x_vars: Dict, sets: Dict) -> List[List[int]]:
    """Extract routes from binary variables."""
    routes = []

    for k in sets['Bs']:
        route = [k]
        current = k
        visited = set([k])

        for _ in range(len(sets['V']) + 2):
            next_node = None
            for key, var in x_vars.items():
                if len(key) == 3 and key[0] == k and key[1] == current:
                    try:
                        if var.X > 0.5:
                            next_node = key[2]
                            break
                    except Exception:
                        continue

            if next_node is None:
                break

            if next_node in visited and next_node not in sets['Be']:
                break

            visited.add(next_node)
            route.append(next_node)

            if next_node in sets['Be']:
                break

            current = next_node

        if len(route) > 2:
            routes.append(route)

    return routes

In [11]:
def initialize_mass_ratios(params: Parameters, sets: Dict, node_to_body: Dict) -> Tuple[Dict, Dict]:
    """
    Initialize mass ratios per paper Section IV.A.

    Uses a coarse grid scan to identify good launch windows, then refines
    the best point with L-BFGS-B. This is more robust than a single
    trust-region solve from one starting guess, which can miss good windows
    on some body pairs (e.g. Earth->FG3) due to local-minima sensitivity.

    Paper intent: "solve the trajectory optimization problem for each pair
    of bodies to find optimal departure and transfer times by using the zero
    departure time and the Hohmann transfer time as the initial guess."
    The grid scan honours this by covering the zero-departure region and
    Hohmann-neighbourhood, then refining.
    """
    print("Initializing mass ratios (per paper Section IV.A)...")

    traj_opt = TrajectoryOptimizer(params)
    mass_ratios = {}

    all_source = sets['Bs'] + sets['V']
    all_dest   = sets['V'] + list(set(sets['Be']))

    body_pair_cache = {}
    body_pair_times = {}  # (body_i.name, body_j.name) -> (best_T_d, best_T_t)
    init_times = {}       # (node_i, node_j) -> (best_T_d, best_T_t)
    eps = 1e-5

    # Coarse grid: 0..13 TU departure × 1,3,5,7,9,11,13 TU transfer
    T_d_grid = np.arange(0.0, 14.0, 1.0)
    T_t_grid = np.arange(1.0, 14.0, 2.0)

    for i in all_source:
        for j in all_dest:
            if i == j:
                continue

            body_i = node_to_body[i]
            body_j = node_to_body[j]

            if body_i.name == body_j.name:
                # Same physical body: arc is forbidden in build_milp so skip entirely.
                # Assigning mr=0.999 here was a modeling error — it made same-body
                # virtual-node hops look free and corrupted the MILP's route choices.
                continue

            pair_key = (body_i.name, body_j.name)
            if pair_key in body_pair_cache:
                mass_ratios[(i, j)] = body_pair_cache[pair_key]
                init_times[(i, j)] = body_pair_times[pair_key]
                continue

            # Step 1: coarse grid — find best launch window
            best_dv   = 1e6
            best_td, best_tt = 0.0, 1.0
            for T_d in T_d_grid:
                for T_t in T_t_grid:
                    try:
                        dv = traj_opt.compute_delta_v(body_i, body_j, T_d, T_t)
                        if np.isfinite(dv) and dv < best_dv:
                            best_dv  = dv
                            best_td, best_tt = T_d, T_t
                    except Exception:
                        continue

            # Step 2: refine from best grid point with L-BFGS-B
            if best_dv < 50.0:
                def objective(x):
                    T_d, T_t = x
                    if T_t < eps:
                        return 1e6
                    try:
                        return traj_opt.compute_delta_v(body_i, body_j, T_d, T_t)
                    except Exception:
                        return 1e6

                try:
                    res = minimize(objective, [best_td, best_tt],
                                   method='L-BFGS-B',
                                   bounds=[(0.0, None), (eps, None)],
                                   options={'maxiter': 200, 'ftol': 1e-10})
                    if np.isfinite(res.fun) and res.fun < best_dv:
                        best_dv = res.fun
                        best_td, best_tt = float(res.x[0]), float(res.x[1])
                except Exception:
                    pass

            if np.isfinite(best_dv) and 0 < best_dv < 50.0:
                mr = np.exp(-best_dv / (params.g0 * params.I_sp))  # km/s, g0 km/s^2 → consistent
                mass_ratios[(i, j)] = float(np.clip(mr, 1e-4, 0.999))
            else:
                mass_ratios[(i, j)] = 0.05

            init_times[(i, j)] = (best_td, best_tt)
            body_pair_cache[pair_key] = mass_ratios[(i, j)]
            body_pair_times[pair_key] = (best_td, best_tt)

    valid_count = sum(1 for mr in mass_ratios.values() if np.isfinite(mr) and 0 < mr <= 1)
    print(f"  Initialized {len(mass_ratios)} transfers ({valid_count} valid)")

    mr_values = [v for v in mass_ratios.values() if v < 0.99]
    if mr_values:
        print(f"  Mass ratio range (excl same-body): [{min(mr_values):.4f}, {max(mr_values):.4f}]")

    return mass_ratios, init_times

## Iterative MILP-NLP Solver

In [12]:
def solve_vrtpp_pr(params: Parameters, sets: Dict, node_to_body: Dict,
                   node_to_name: Dict, max_iterations: int = 50,
                   convergence_tol: float = 1e-3) -> Dict:
    """
    Complete iterative MILP-NLP algorithm.

    Returns solution dict with routes, times, delta-v, etc.
    """

    print("=" * 80)
    print("STARTING VRTPP-PR OPTIMIZATION")
    print("=" * 80)

    # Initialize trajectory optimizer
    traj_opt = TrajectoryOptimizer(params)

    # Step 1: Initialize mass ratios
    mass_ratios, init_times = initialize_mass_ratios(params, sets, node_to_body)
    delta_v_matrix = {}
    departure_times = {}
    transfer_times = {}
    arc_results = {}      # (i,j) -> last NLP result; used for warm-start (Fix 1)

    # Debug: print mass ratios for critical arcs
    print("\nCritical mass ratios (Earth->FG3->Bennu->Earth):")
    for (i, j), mr in sorted(mass_ratios.items()):
        bi = node_to_body[i].name if i in node_to_body else "?"
        bj = node_to_body[j].name if j in node_to_body else "?"
        dv_est = -np.log(max(mr, 1e-10)) * params.g0 * params.I_sp  # km/s (g0 in km/s^2)
        if ("Earth" in bi and "FG3" in bj) or \
           ("FG3" in bi and "Bennu" in bj) or \
           ("Bennu" in bi and "Earth" in bj):
            print(f"  ({i:2d},{j:2d}) {bi:20s} -> {bj:20s}: mr={mr:.4f}, dv~{dv_est:.1f} km/s")

    warm_start = None
    prev_routes = None
    stable_route_iters = 0   # consecutive iterations with unchanged route (by body names)
    start_time = time.time()

    # Track consecutive iterations with no routes for early termination
    consecutive_no_routes = 0

    for iteration in range(max_iterations):
        print(f"\n{'=' * 80}")
        print(f"ITERATION {iteration + 1}")
        print(f"{'=' * 80}")

        # Step 2: Solve MILP with fixed mass ratios
        print("\n[MILP] Building model...")
        model, variables = build_milp(params, sets, mass_ratios, node_to_name, node_to_body)

        if warm_start:
            for key, val in warm_start.items():
                if key in variables['x']:
                    variables['x'][key].Start = val

        model.setParam('TimeLimit', 100.0)  # paper uses 100s (Intel Core Ultra 9 285K); raise if needed on slower hardware
        print("[MILP] Solving...")
        model.optimize()

        if model.Status == GRB.INFEASIBLE:
            print(f"[MILP] Infeasible (status {model.Status})")
            if iteration == 0:
                print("No feasible solution!")
                return None
            else:
                print("Using previous solution")
                break
        elif model.Status not in [GRB.OPTIMAL, GRB.TIME_LIMIT, GRB.SUBOPTIMAL]:
            print(f"[MILP] Unexpected status {model.Status}")
            if iteration == 0:
                return None
            break

        if model.SolCount == 0:
            print(f"[MILP] No solution found (status {model.Status})")
            if iteration == 0:
                return None
            break

        if model.Status == GRB.TIME_LIMIT:
            print(f"[MILP] Time limit reached, using best solution (gap: {model.MIPGap*100:.1f}%)")

        print(f"[MILP] Objective: {model.ObjVal:.4f}")

        # Extract routes
        routes = extract_routes(variables['x'], sets)
        print(f"[MILP] Routes: {len(routes)} spacecraft")

        if len(routes) == 0:
            consecutive_no_routes += 1
            print(f"[MILP] No routes found ({consecutive_no_routes} consecutive)")
            if consecutive_no_routes >= 2:
                print("[MILP] Early termination: no routes for 2 consecutive iterations")
                break
            continue
        else:
            consecutive_no_routes = 0

        for i, route in enumerate(routes):
            route_names = [node_to_name[n] for n in route]
            print(f"  Spacecraft {i + 1}: {' -> '.join(route_names)}")

        # Step 3: Optimize trajectories (NLP)
        print("\n[NLP] Optimizing trajectories...")
        old_dv_matrix = delta_v_matrix.copy()

        # Build current arc set (used by Fix 3 active-arc convergence)
        current_arc_set = set()
        for spacecraft_route in routes:
            for k in range(len(spacecraft_route) - 1):
                current_arc_set.add((spacecraft_route[k], spacecraft_route[k + 1]))

        for spacecraft_route in routes:
            T_arrival = 0.0

            for k in range(len(spacecraft_route) - 1):
                node_i = spacecraft_route[k]
                node_j = spacecraft_route[k + 1]
                arc = (node_i, node_j)

                body_i = node_to_body[node_i]
                body_j = node_to_body[node_j]

                # Fix 1: pass previous T_d and T_t as warm-start
                prev_result = arc_results.get(arc)
                T_d_prev_val = prev_result['T_d'] if prev_result is not None else None
                T_t_prev_val = prev_result['T_t'] if prev_result is not None else None

                print(f"  Optimizing {node_to_name[node_i]} -> {node_to_name[node_j]}...", end=" ")
                result = traj_opt.optimize_segment(body_i, body_j, T_arrival,
                                                   T_t_prev=T_t_prev_val,
                                                   T_d_prev=T_d_prev_val)

                arc_results[arc] = result
                departure_times[arc] = result['T_d']
                transfer_times[arc] = result['T_t']
                delta_v_matrix[arc] = result['delta_v']
                mass_ratios[arc] = result['mass_ratio']
                T_arrival = result['T_a']

                print(f"dv={result['delta_v']:.2f} km/s, T_d={result['T_d']:.2f} TU, T_t={result['T_t']:.2f} TU")


        # Step 4: Check convergence (Equation 47)
        if iteration > 0:
            # Compare routes by physical body names, independent of spacecraft
            # label order (Gurobi can return same routes with SC1/SC2 swapped,
            # which previously counted as "changed" and wasted an iteration).
            def route_body_sig(rts):
                return frozenset(
                    tuple(node_to_body[n].name for n in r) for r in rts
                )
            route_changed = (prev_routes is None or
                             route_body_sig(routes) != route_body_sig(prev_routes))

            if route_changed:
                stable_route_iters = 0
                print(f"\n[CONVERGENCE] Route changed, continuing...")
            else:
                stable_route_iters += 1
                if old_dv_matrix:
                    # Fix 3: restrict convergence check to active route arcs only.
                    diff_sq = 0.0
                    max_dv_old = 0.0
                    for arc in current_arc_set:
                        dv_new = delta_v_matrix.get(arc, 0.0)
                        dv_old = old_dv_matrix.get(arc, 0.0)
                        diff_sq += (dv_new - dv_old) ** 2
                        max_dv_old = max(max_dv_old, dv_old)
                    change = np.sqrt(diff_sq) / max_dv_old if max_dv_old > 0 else 0.0

                    print(f"\n[CONVERGENCE] Active-arc dv change: {change:.6f} (tol: {convergence_tol}, stable iters: {stable_route_iters})")

                    if change < convergence_tol:
                        print(f"\n{'=' * 80}")
                        print(f"CONVERGED after {iteration + 1} iterations!")
                        print(f"{'=' * 80}")
                        break

                    # Soft convergence: route unchanged for 5+ consecutive iterations
                    # AND dv change is small (< 0.05). The NLP has multiple local
                    # minima for some arcs and oscillates between them by ~0.04,
                    # preventing the strict 0.001 threshold from ever triggering.
                    # Route stability is a stronger convergence signal in that case.
                    if stable_route_iters >= 5 and change < 0.05:
                        print(f"\n{'=' * 80}")
                        print(f"CONVERGED (soft) after {iteration + 1} iterations!")
                        print(f"  Route stable for {stable_route_iters} consecutive iterations, dv change={change:.4f}")
                        print(f"{'=' * 80}")
                        break

        prev_routes = [r[:] for r in routes]

        # Warm start for next iteration
        warm_start = {}
        for k, v in variables['x'].items():
            try:
                if v.X > 0.5:
                    warm_start[k] = v.X
            except Exception:
                continue

    elapsed = time.time() - start_time

    # Extract numerical values from Gurobi variables before model goes out of scope
    u_values = {}
    r_values = {}
    q_values = {}
    if model.SolCount > 0:
        for key, var in variables['u'].items():
            try:
                u_values[key] = var.X
            except Exception:
                u_values[key] = 0.0
        for key, var in variables['r'].items():
            try:
                r_values[key] = var.X
            except Exception:
                r_values[key] = 0.0
        for key, var in variables['q'].items():
            try:
                q_values[key] = var.X
            except Exception:
                q_values[key] = 0.0

    # Final solution
    solution = {
        'status': 'converged' if iteration < max_iterations - 1 else 'max_iterations',
        'iterations': iteration + 1,
        'elapsed_time': elapsed,
        'objective': model.ObjVal if model.SolCount > 0 else 0.0,
        'routes': routes,
        'departure_times': departure_times,
        'transfer_times': transfer_times,
        'delta_v_matrix': delta_v_matrix,
        'mass_ratios': mass_ratios,
        'u_values': u_values,
        'r_values': r_values,
        'q_values': q_values
    }

    return solution

## Run 10 Instances

## Config

In [13]:
# ── EXPERIMENT CONFIG ── change N_R and N_M to run a different experiment
N_R         = 1    # number of refueling asteroids
N_M         = 6    # number of mining asteroids
N_INSTANCES = 10   # problem instances per configuration
BASE_SEED   = 42   # random seed for first instance

## Run Experiment

In [14]:
BASE_SEED = 42

results = []

for instance_idx in range(N_INSTANCES):
    seed = BASE_SEED + instance_idx
    sep = "=" * 60
    print(sep)
    print("Instance " + str(instance_idx+1) + "/" + str(N_INSTANCES) + "  (seed=" + str(seed) + ")")
    print(sep)

    rand_refueling, rand_mining = generate_random_asteroids(N_R, N_M, seed=seed)

    instance_params = Parameters()
    instance_sets = build_index_sets(instance_params, n_refuel=N_R, n_mine=N_M)
    instance_node_to_body, instance_node_to_name = build_node_mapping(
        instance_sets, rand_refueling, rand_mining
    )

    sol = solve_vrtpp_pr(
        params=instance_params,
        sets=instance_sets,
        node_to_body=instance_node_to_body,
        node_to_name=instance_node_to_name,
        max_iterations=50,
        convergence_tol=1e-3
    )

    if sol is None:
        print("  Instance " + str(instance_idx+1) + ": no solution")
        results.append({
            "instance": instance_idx+1, "seed": seed,
            "iterations": 50, "time": 0.0,
            "mining_count": 0, "trivial": 0, "non_converged": 1, "status": "failed"
        })
        continue

    mining_count = sum(1 for r in sol["routes"] for n in r if n in instance_sets["M"])
    trivial = 1 if mining_count == 0 else 0
    non_conv = 1 if sol["status"] == "max_iterations" else 0

    results.append({
        "instance": instance_idx+1, "seed": seed,
        "iterations": sol["iterations"],
        "time": sol["elapsed_time"],
        "mining_count": mining_count,
        "trivial": trivial,
        "non_converged": non_conv,
        "status": sol["status"]
    })
    print("  -> " + sol["status"] + ", " + str(sol["iterations"]) + " iters, " + str(round(sol["elapsed_time"],1)) + "s, " + str(mining_count) + " mining asteroids")

Instance 1/10  (seed=42)
STARTING VRTPP-PR OPTIMIZATION
Initializing mass ratios (per paper Section IV.A)...


  Initialized 138 transfers (138 valid)
  Mass ratio range (excl same-body): [0.0043, 0.5651]

Critical mass ratios (Earth->FG3->Bennu->Earth):

ITERATION 1

[MILP] Building model...
Restricted license - for non-production use only - expires 2027-11-29


[MILP] Solving...


[MILP] Objective: 38.9552
[MILP] Routes: 2 spacecraft
  Spacecraft 1: Earth -> R1 -> M4 -> R1 -> M5 -> M1 -> R1 -> Earth
  Spacecraft 2: Earth -> M2 -> Earth

[NLP] Optimizing trajectories...
  Optimizing Earth -> R1... dv=19.95 km/s, T_d=0.00 TU, T_t=5.73 TU
  Optimizing R1 -> M4... 

dv=11.22 km/s, T_d=5.80 TU, T_t=19.74 TU
  Optimizing M4 -> R1... dv=22.67 km/s, T_d=25.61 TU, T_t=9.52 TU
  Optimizing R1 -> M5... 

...

dv=4.60 km/s, T_d=14.88 TU, T_t=3.39 TU
  Optimizing M3 -> Earth... dv=9.47 km/s, T_d=22.07 TU, T_t=13.44 TU

[CONVERGENCE] Active-arc dv change: 0.002800 (tol: 0.001, stable iters: 4)

ITERATION 32

[MILP] Building model...
[MILP] Solving...


[MILP] Objective: 18.8770
[MILP] Routes: 2 spacecraft
  Spacecraft 1: Earth -> R1 -> M4 -> Earth
  Spacecraft 2: Earth -> R1 -> M3 -> Earth

[NLP] Optimizing trajectories...
  Optimizing Earth -> R1... dv=5.78 km/s, T_d=4.92 TU, T_t=5.01 TU
  Optimizing R1 -> M4... 

dv=9.61 km/s, T_d=14.15 TU, T_t=30.00 TU
  Optimizing M4 -> Earth... dv=7.60 km/s, T_d=47.74 TU, T_t=6.26 TU
  Optimizing Earth -> R1... 

dv=5.78 km/s, T_d=4.90 TU, T_t=4.98 TU
  Optimizing R1 -> M3... 

dv=4.60 km/s, T_d=14.85 TU, T_t=3.41 TU
  Optimizing M3 -> Earth... dv=9.47 km/s, T_d=22.07 TU, T_t=13.43 TU

[CONVERGENCE] Active-arc dv change: 0.002815 (tol: 0.001, stable iters: 5)

CONVERGED (soft) after 32 iterations!
  Route stable for 5 consecutive iterations, dv change=0.0028
  -> converged, 32 iters, 58.3s, 2 mining asteroids


## Summary Statistics

In [15]:
import statistics

iters = [r["iterations"] for r in results]
times = [r["time"]       for r in results]
mines = [r["mining_count"] for r in results]
trivials  = sum(r["trivial"]       for r in results)
non_convs = sum(r["non_converged"] for r in results)

sep = "=" * 70
print(sep)
print("RESULTS: n_r=" + str(N_R) + ", n_m=" + str(N_M) + "  (" + str(N_INSTANCES) + " instances)")
print(sep)
print("{:<30s} {:>8s} {:>8s} {:>8s}".format("", "Min", "Max", "Mean"))
print("{:<30s} {:>8d} {:>8d} {:>8.1f}".format("Iterations", min(iters), max(iters), statistics.mean(iters)))
print("{:<30s} {:>8.2f} {:>8.2f} {:>8.2f}".format("Time (s)", min(times), max(times), statistics.mean(times)))
print("{:<30s} {:>8d} {:>8d} {:>8.1f}".format("Mining asteroids", min(mines), max(mines), statistics.mean(mines)))
print("Trivial problems:       " + str(trivials))
print("Non-converged problems: " + str(non_convs))
print()
print("Paper reference (n_r=1, n_m=4):")
print("  Iter: min=3, max=24, mean=11.6")
print("  Time: min=0.68s, max=10.36s, mean=4.89s")
print("  Mine: min=1, max=3, mean=1.9  | Trivial=2, Non-conv=1")
print()
row = ("our_model," + str(N_R) + "," + str(N_M) + ","
       + str(min(iters)) + "," + str(max(iters)) + "," + str(round(statistics.mean(iters),1)) + ","
       + str(round(min(times),2)) + "," + str(round(max(times),2)) + "," + str(round(statistics.mean(times),2)) + ","
       + str(min(mines)) + "," + str(max(mines)) + "," + str(round(statistics.mean(mines),1)) + ","
       + str(trivials) + "," + str(non_convs) + ",10 random instances seed 42-51")
print("--- results.csv row ---")
print(row)
print()
print("{:>5s} {:>6s} {:>6s} {:>8s} {:>5s} {}".format("Inst", "Seed", "Iters", "Time(s)", "Mine", "Status"))
for r in results:
    print("{:>5d} {:>6d} {:>6d} {:>8.2f} {:>5d} {}".format(
        r["instance"], r["seed"], r["iterations"], r["time"], r["mining_count"], r["status"]))

RESULTS: n_r=1, n_m=6  (10 instances)
                                    Min      Max     Mean
Iterations                            2       35     22.7
Time (s)                          18.53   106.31    75.84
Mining asteroids                      1        5      3.8
Trivial problems:       0
Non-converged problems: 0

Paper reference (n_r=1, n_m=4):
  Iter: min=3, max=24, mean=11.6
  Time: min=0.68s, max=10.36s, mean=4.89s
  Mine: min=1, max=3, mean=1.9  | Trivial=2, Non-conv=1

--- results.csv row ---
our_model,1,6,2,35,22.7,18.53,106.31,75.84,1,5,3.8,0,0,10 random instances seed 42-51

 Inst   Seed  Iters  Time(s)  Mine Status
    1     42     14    25.12     3 converged
    2     43      2    18.53     5 converged
    3     44     19    75.00     5 converged
    4     45     35    99.23     5 converged
    5     46     30   106.31     4 converged
    6     47     20    92.94     4 converged
    7     48     28   102.09     5 converged
    8     49     19   105.25     4 converged
